# 4. Integración y Construcción de Datasets Maestros

En esta sección el objetivo es producir **4 datasets maestros** listos para la Fase 2 (EDA) y
Fase 3 (Modelado ML), eliminando la necesidad de hacer joins ad-hoc en cada notebook.

| Join | Decisión | Implementación |
|---|---|---|
| JOIN-1 Agrícola | gap temporal → precio histórico como feature | `precio_hist_cultivo` como promedio por pais×cultivo |
| JOIN-2 Ganadero | subsidios → flag binario | `tiene_subsidio_activo` (0/1) |
| JOIN-3 Precios | sentimiento agregado país×mes | `sentiment_score_medio` por pais×anio×mes |
| JOIN-4 Políticas | Contexto estático + `normativa_vigente` temporal | Flag 0/1 según `anio_implementacion` |
| JOIN-5 Imágenes | Soporte de calidad: cruzar con eventos extremos | `imagen_durante_evento_extremo` (0/1) |

**Nota importante — P4 standalone**
`master_precios.csv` es el único dataset maestro que **no se une** con
`produccion_agricola`. P4 (Predicción de Precios) se modela como serie temporal
independiente. El gap 2011-2020 (producción) vs 2021+ (precios) hace inviable
el join temporal directo.

## Bloque PRE - Agregaciones previas obligatorias

Antes de ejecutar ningún join maestro, 4 fuentes requieren ser agregadas
o pivotadas porque su granularidad es más fina que las claves de unión.

| ID | Dataset origen | Granularidad original | Granularidad objetivo | Motivo |
|---|---|---|---|---|
| PRE-4.1 | condiciones_climaticas | pais × **region** × anio | pais × anio | Múltiples regiones por país |
| PRE-4.2 | reportes_plagas | evento puntual (fecha) | pais × **cultivo** × anio | Múltiples eventos por combinación |
| PRE-4.3 | noticias_processed | noticia (fecha) | pais × anio × **mes** | Relación M:N país/cultivo por noticia |
| PRE-4.4 | politicas_* (3 archivos) | pais × tipo | pais (lookup) + pais × anio (flag) | Necesitan pivot a formato wide |

**Estas 4 agregaciones se almacenan en memoria (DataFrames).
No se exportan como CSV intermedios** — solo los 4 datasets maestros
finales se persisten en `data/processed/`.

### Bloque PRE 4.1. Clima agregado → clima_pais_anio

In [1]:
import pandas as pd
import numpy as np
import os

# ── Carga ──────────────────────────────────────────────────────────────────
df_clim = pd.read_csv('../../data/processed/condiciones_climaticas_processed.csv')
print(f"📥 condiciones_climaticas_processed cargado: {df_clim.shape}")
print(f"   Regiones únicas por país:")
print(df_clim.groupby('pais')['region'].nunique().to_string())

# ── Estrategia de agregación ───────────────────────────────────────────────
# Variables continuas  → promedio aritmético entre regiones del mismo país/año
# Variables de eventos → suma (n_eventos totales) y max (peor evento del año)
# Variables de días/meses → promedio entre regiones

cols_promedio = [
    'temperatura_promedio', 'temperatura_maxima', 'temperatura_minima',
    'precipitacion_total', 'humedad_relativa_promedio',
    'dias_con_heladas', 'meses_estres_hidrico', 'indice_aridez'
]

cols_suma = [
    'n_eventos',
    'ev_granizo', 'ev_helada_tardia', 'ev_incendios', 'ev_inundacion',
    'ev_ola_calor', 'ev_sequia_extrema', 'ev_sequia_moderada',
    'ev_sequia_severa', 'ev_tormenta_severa'
]

cols_max = ['n_eventos']   # también como max para saber si hubo al menos 1 región afectada

# Construir diccionario de agregaciones
agg_dict = {col: 'mean' for col in cols_promedio}
agg_dict.update({col: 'sum' for col in cols_suma})
# Renombrar n_eventos_sum al final → n_eventos_regiones_afectadas
agg_dict['n_eventos'] = 'sum'

clima_pais_anio = (
    df_clim
    .groupby(['pais', 'codigo_iso', 'anio'], as_index=False)
    .agg(agg_dict)
)

# Renombrar n_eventos sumado para mayor claridad
clima_pais_anio.rename(columns={'n_eventos': 'n_eventos_total_regiones'}, inplace=True)

# ── Columna resumen: país con al menos 1 evento extremo ese año ────────────
cols_eventos_bin = [
    'ev_granizo', 'ev_helada_tardia', 'ev_incendios', 'ev_inundacion',
    'ev_ola_calor', 'ev_sequia_extrema', 'ev_sequia_moderada',
    'ev_sequia_severa', 'ev_tormenta_severa'
]
clima_pais_anio['flag_anio_eventos_extremos'] = (
    clima_pais_anio[cols_eventos_bin].sum(axis=1) > 0
).astype(int)

# ── Redondeo para legibilidad ──────────────────────────────────────────────
for col in cols_promedio:
    clima_pais_anio[col] = clima_pais_anio[col].round(2)

# ── Validación ────────────────────────────────────────────────────────────
print(f"\n{'='*52}")
print(f"  [PRE-A] clima_pais_anio — RESULTADO")
print(f"{'='*52}")
print(f"  Shape             : {clima_pais_anio.shape}")
print(f"  Países únicos     : {clima_pais_anio['pais'].nunique()}")
print(f"  Rango temporal    : {clima_pais_anio['anio'].min()} – {clima_pais_anio['anio'].max()}")
print(f"  Registros/pais    : {len(clima_pais_anio) / clima_pais_anio['pais'].nunique():.0f} años de media")
print(f"  Nulos             : {clima_pais_anio.isnull().sum().sum()}")
print(f"  Años con eventos  : {clima_pais_anio['flag_anio_eventos_extremos'].sum()} "
      f"({clima_pais_anio['flag_anio_eventos_extremos'].mean()*100:.1f}%)")
print(f"\n  Vista previa:")
display(clima_pais_anio.head(4))

print(f"\n  ✅ [PRE-A] lista en memoria como: clima_pais_anio")
print(f"     Columnas ({len(clima_pais_anio.columns)}): {list(clima_pais_anio.columns)}")

📥 condiciones_climaticas_processed cargado: (200, 23)
   Regiones únicas por país:
pais
Alemania          1
Argentina         3
Australia         3
Brasil            3
China             1
Estados Unidos    3
Francia           1
India             3
Kenia             1
México            1

  [PRE-A] clima_pais_anio — RESULTADO
  Shape             : (100, 22)
  Países únicos     : 10
  Rango temporal    : 2011 – 2020
  Registros/pais    : 10 años de media
  Nulos             : 0
  Años con eventos  : 46 (46.0%)

  Vista previa:


,pais,codigo_iso,anio,temperatura_promedio,temperatura_maxima,temperatura_minima,precipitacion_total,humedad_relativa_promedio,dias_con_heladas,meses_estres_hidrico,...,ev_granizo,ev_helada_tardia,ev_incendios,ev_inundacion,ev_ola_calor,ev_sequia_extrema,ev_sequia_moderada,ev_sequia_severa,ev_tormenta_severa,flag_anio_eventos_extremos
0,Alemania,DEU,2011,9.9,18.2,1.8,886.0,76.0,72.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,Alemania,DEU,2012,14.6,21.8,6.1,404.0,83.0,116.0,10.0,...,0,0,0,0,0,0,0,0,0,0
2,Alemania,DEU,2013,11.4,21.5,-1.7,731.0,68.0,3.0,4.0,...,0,0,0,0,0,0,0,0,0,0
3,Alemania,DEU,2014,20.5,32.3,9.6,1589.0,50.0,115.0,2.0,...,0,0,0,0,1,0,0,1,0,1



  ✅ [PRE-A] lista en memoria como: clima_pais_anio
     Columnas (22): ['pais', 'codigo_iso', 'anio', 'temperatura_promedio', 'temperatura_maxima', 'temperatura_minima', 'precipitacion_total', 'humedad_relativa_promedio', 'dias_con_heladas', 'meses_estres_hidrico', 'indice_aridez', 'n_eventos_total_regiones', 'ev_granizo', 'ev_helada_tardia', 'ev_incendios', 'ev_inundacion', 'ev_ola_calor', 'ev_sequia_extrema', 'ev_sequia_moderada', 'ev_sequia_severa', 'ev_tormenta_severa', 'flag_anio_eventos_extremos']


### PRE 4.2. Plagas agregadas → plagas_pais_cultivo_anio

In [2]:
df_plagas = pd.read_csv('../../data/processed/reportes_plagas_processed.csv')
print(f"📥 reportes_plagas_processed cargado: {df_plagas.shape}")

# ── Verificar cobertura de países vs datasets maestros ────────────────────
print(f"\n  Países en plagas   : {sorted(df_plagas['pais'].unique())}")
print(f"  Cultivos en plagas : {sorted(df_plagas['cultivo'].unique())}")
print(f"  Rango temporal     : {df_plagas['año'].min()} – {df_plagas['año'].max()}")

# ── Agregación por pais × cultivo × anio ──────────────────────────────────
plagas_agg = (
    df_plagas
    .groupby(['pais', 'cultivo', 'año'], as_index=False)
    .agg(
        n_eventos_plaga        = ('agente',          'count'),
        severidad_max          = ('severidad_num',   'max'),
        severidad_media        = ('severidad_num',   'mean'),
        area_total_ha          = ('area_ha',         'sum'),
        eficacia_media_pct     = ('eficacia_pct',    'mean'),
        perdida_media_pct      = ('perdida_pct',     'mean'),
        perdida_max_pct        = ('perdida_pct',     'max'),
        n_plagas_distintas     = ('agente',          'nunique'),
        n_tratamientos_quimicos= ('tipo_tratamiento',
                                  lambda x: (x == 'Químico').sum()),
    )
)
plagas_agg.rename(columns={'año': 'anio'}, inplace=True)

# Redondeo
for col in ['severidad_media', 'eficacia_media_pct',
            'perdida_media_pct', 'perdida_max_pct']:
    plagas_agg[col] = plagas_agg[col].round(2)

# ── Validación ────────────────────────────────────────────────────────────
print(f"\n{'='*52}")
print(f"  [PRE-B] plagas_pais_cultivo_anio — RESULTADO")
print(f"{'='*52}")
print(f"  Shape             : {plagas_agg.shape}")
print(f"  Países únicos     : {plagas_agg['pais'].nunique()}")
print(f"  Cultivos únicos   : {plagas_agg['cultivo'].nunique()}")
print(f"  Rango temporal    : {plagas_agg['anio'].min()} – {plagas_agg['anio'].max()}")
print(f"  Nulos             : {plagas_agg.isnull().sum().sum()}")
print(f"\n  Eventos por combinación pais×cultivo×anio:")
print(f"    min={plagas_agg['n_eventos_plaga'].min()} | "
      f"max={plagas_agg['n_eventos_plaga'].max()} | "
      f"media={plagas_agg['n_eventos_plaga'].mean():.2f}")
print(f"\n  Cobertura de cultivos (top 5 combinaciones):")
print(plagas_agg.groupby('cultivo')['n_eventos_plaga']
      .agg(['count','sum'])
      .rename(columns={'count':'combinaciones_pais_anio','sum':'eventos_total'})
      .sort_values('eventos_total', ascending=False)
      .head(5).to_string())
print(f"\n  Vista previa:")
display(plagas_agg.head(4))
print(f"\n  ✅ [PRE-B] lista en memoria como: plagas_agg")
print(f"     Columnas ({len(plagas_agg.columns)}): {list(plagas_agg.columns)}")

📥 reportes_plagas_processed cargado: (100, 15)

  Países en plagas   : ['Alemania', 'Argentina', 'Australia', 'Brasil', 'China', 'Estados Unidos', 'Francia', 'India']
  Cultivos en plagas : ['Algodón', 'Arroz', 'Café', 'Caña de azúcar', 'Cebada', 'Girasol', 'Maíz', 'Soja', 'Trigo', 'Té']
  Rango temporal     : 2020 – 2022

  [PRE-B] plagas_pais_cultivo_anio — RESULTADO
  Shape             : (75, 12)
  Países únicos     : 8
  Cultivos únicos   : 10
  Rango temporal    : 2020 – 2022
  Nulos             : 0

  Eventos por combinación pais×cultivo×anio:
    min=1 | max=3 | media=1.33

  Cobertura de cultivos (top 5 combinaciones):
                combinaciones_pais_anio  eventos_total
cultivo                                               
Arroz                                11             19
Maíz                                 10             15
Caña de azúcar                       10             12
Soja                                  7              9
Trigo                              

,pais,cultivo,anio,n_eventos_plaga,severidad_max,severidad_media,area_total_ha,eficacia_media_pct,perdida_media_pct,perdida_max_pct,n_plagas_distintas,n_tratamientos_quimicos
0,Alemania,Algodón,2021,1,3,3.0,1734.0,74.0,11.0,11.0,1,1
1,Alemania,Arroz,2020,2,3,2.5,3761.0,78.5,28.0,39.0,2,1
2,Alemania,Arroz,2021,2,3,2.5,3650.0,79.0,34.0,40.0,2,0
3,Alemania,Café,2020,1,1,1.0,207.0,88.0,13.0,13.0,1,1



  ✅ [PRE-B] lista en memoria como: plagas_agg
     Columnas (12): ['pais', 'cultivo', 'anio', 'n_eventos_plaga', 'severidad_max', 'severidad_media', 'area_total_ha', 'eficacia_media_pct', 'perdida_media_pct', 'perdida_max_pct', 'n_plagas_distintas', 'n_tratamientos_quimicos']


### PRE 4.3. Noticias agregadas → noticias_pais_mes

In [3]:
df_noticias = pd.read_csv('../../data/processed/noticias_processed.csv')
print(f"📥 noticias_processed cargado: {df_noticias.shape}")
print(f"   Rango temporal: {df_noticias['fecha'].min()} → {df_noticias['fecha'].max()}")
print(f"   Países en paises_afectados_str (muestra): "
      f"{df_noticias['paises_afectados_str'].head(3).tolist()}")

# ── Explotar la relación M:N país × noticia ────────────────────────────────
# Cada noticia puede afectar a N países (separados por '|')
# Decisión arquitecto JOIN-3 Opción A: agregar a nivel país × mes
# (sin desagregar por cultivo)

df_noticias_exp = df_noticias.copy()
df_noticias_exp['paises_lista'] = df_noticias_exp['paises_afectados_str'].str.split('|')
df_noticias_exp = df_noticias_exp.explode('paises_lista')
df_noticias_exp['pais'] = df_noticias_exp['paises_lista'].str.strip()

print(f"\n   Filas tras explode: {len(df_noticias_exp)} "
      f"(de {len(df_noticias)} originales)")
print(f"   Países únicos en noticias: {sorted(df_noticias_exp['pais'].unique())}")

# ── Agregación por pais × anio × mes ──────────────────────────────────────
noticias_pais_mes = (
    df_noticias_exp
    .groupby(['pais', 'año', 'mes'], as_index=False)
    .agg(
        n_noticias_total      = ('id_noticia',         'count'),
        sentiment_score_medio = ('sentiment_compound', 'mean'),
        n_noticias_negativas  = ('sentiment_label',
                                  lambda x: (x == 'negativo').sum()),
        n_noticias_positivas  = ('sentiment_label',
                                  lambda x: (x == 'positivo').sum()),
        n_noticias_clima      = ('categoria',
                                  lambda x: (x == 'clima').sum()),
        n_noticias_mercado    = ('categoria',
                                  lambda x: (x == 'mercado').sum()),
        severidad_max_noticia = ('severidad',
                                  lambda x: x.map({'baja':1,'media':2,'alta':3})
                                             .max()),
    )
)
noticias_pais_mes.rename(columns={'año': 'anio'}, inplace=True)
noticias_pais_mes['sentiment_score_medio'] = (
    noticias_pais_mes['sentiment_score_medio'].round(4)
)

# ── Validación ────────────────────────────────────────────────────────────
print(f"\n{'='*52}")
print(f"  [PRE-C] noticias_pais_mes — RESULTADO")
print(f"{'='*52}")
print(f"  Shape             : {noticias_pais_mes.shape}")
print(f"  Países únicos     : {noticias_pais_mes['pais'].nunique()}")
print(f"  Rango temporal    : {noticias_pais_mes['anio'].min()} – "
      f"{noticias_pais_mes['anio'].max()}")
print(f"  Nulos             : {noticias_pais_mes.isnull().sum().sum()}")

print(f"\n  Noticias por mes (media): "
      f"{noticias_pais_mes['n_noticias_total'].mean():.2f}")
print(f"  Distribución sentiment_score_medio:")
print(f"    min={noticias_pais_mes['sentiment_score_medio'].min():.3f} | "
      f"max={noticias_pais_mes['sentiment_score_medio'].max():.3f} | "
      f"media={noticias_pais_mes['sentiment_score_medio'].mean():.3f}")

print(f"\n  Cobertura pais × mes respecto a precios_mercado:")
df_pre = pd.read_csv('../../data/processed/precios_mercado_processed.csv')
precios_pais_mes = df_pre[['pais','anio','mes']].drop_duplicates()
merge_check = precios_pais_mes.merge(
    noticias_pais_mes[['pais','anio','mes']],
    on=['pais','anio','mes'], how='left', indicator=True
)
n_con = (merge_check['_merge'] == 'both').sum()
n_sin = (merge_check['_merge'] == 'left_only').sum()
print(f"    Combinaciones pais×mes en precios : {len(precios_pais_mes)}")
print(f"    Con cobertura de noticias          : {n_con} ({n_con/len(precios_pais_mes)*100:.1f}%)")
print(f"    Sin cobertura (NaN tras join)      : {n_sin} ({n_sin/len(precios_pais_mes)*100:.1f}%)")

print(f"\n  Vista previa:")
display(noticias_pais_mes.head(4))
print(f"\n  ✅ [PRE-C] lista en memoria como: noticias_pais_mes")
print(f"     Columnas ({len(noticias_pais_mes.columns)}): "
      f"{list(noticias_pais_mes.columns)}")

📥 noticias_processed cargado: (60, 14)
   Rango temporal: 2021-01-01 → 2023-07-28
   Países en paises_afectados_str (muestra): ['Australia', 'India|Argentina|Rusia', 'Argentina|México']

   Filas tras explode: 113 (de 60 originales)
   Países únicos en noticias: ['Alemania', 'Argentina', 'Australia', 'Brasil', 'Canadá', 'China', 'España', 'Estados Unidos', 'Francia', 'India', 'Italia', 'Kenia', 'México', 'Nueva Zelanda', 'Rusia']

  [PRE-C] noticias_pais_mes — RESULTADO
  Shape             : (98, 10)
  Países únicos     : 15
  Rango temporal    : 2021 – 2023
  Nulos             : 0

  Noticias por mes (media): 1.15
  Distribución sentiment_score_medio:
    min=-0.900 | max=0.750 | media=0.377

  Cobertura pais × mes respecto a precios_mercado:
    Combinaciones pais×mes en precios : 288
    Con cobertura de noticias          : 66 (22.9%)
    Sin cobertura (NaN tras join)      : 222 (77.1%)

  Vista previa:


,pais,anio,mes,n_noticias_total,sentiment_score_medio,n_noticias_negativas,n_noticias_positivas,n_noticias_clima,n_noticias_mercado,severidad_max_noticia
0,Alemania,2021,9,1,0.75,0,1,0,0,3
1,Alemania,2022,5,1,0.75,0,1,0,0,3
2,Alemania,2022,9,1,0.45,0,1,0,1,2
3,Alemania,2022,10,1,0.75,0,1,0,0,1



  ✅ [PRE-C] lista en memoria como: noticias_pais_mes
     Columnas (10): ['pais', 'anio', 'mes', 'n_noticias_total', 'sentiment_score_medio', 'n_noticias_negativas', 'n_noticias_positivas', 'n_noticias_clima', 'n_noticias_mercado', 'severidad_max_noticia']


###  Alerta de cobertura — PRE-4.3. (Noticias / Sentimiento)

La cobertura de sentimiento tras el join con `precios_mercado` es del **22.9%**:
solo 66 de 288 combinaciones `pais × mes` tendrán dato de sentimiento.

**Causas identificadas:**
1. El dataset de noticias tiene solo 60 noticias para 3 años (2021–2023)
   → media de ~1.7 noticias/mes globales, muy pocas por país
2. La distribución es irregular: algunos meses/países concentran varias
   noticias y muchos no tienen ninguna

**Estrategia de imputación aprobada para master_precios:**
- `sentiment_score_medio` → imputar con **0.0** (neutral)
  Justificación: ausencia de noticia = ausencia de señal, no señal negativa.
  Valor 0 en la escala = sentimiento neutro.
- `n_noticias_*` → imputar con **0** (conteos)
- `severidad_max_noticia` → imputar con **0** (sin noticia = sin severidad)
- Crear flag: `flag_sin_noticias` (1 = fila sin cobertura de sentimiento)

**Implicación para P4 (Modelado):**
Con 77.1% de NaN imputados a 0, la feature de sentimiento tendrá
**baja varianza efectiva**. En Fase 3 se deberá evaluar su importancia
real con feature importance antes de incluirla en el modelo final.

**Países descartados del join** (presentes en noticias, ausentes en precios):
Canadá, Rusia, Italia → se pierden silenciosamente en el LEFT JOIN. Correcto.

### PRE 4.4.  Lookup de políticas → 3 tablas

In [4]:
df_sub  = pd.read_csv('../../data/processed/politicas_subsidios.csv')
df_reg  = pd.read_csv('../../data/processed/politicas_regulaciones.csv')
df_acu  = pd.read_csv('../../data/processed/politicas_acuerdos.csv')

print(f"📥 politicas_subsidios    : {df_sub.shape}")
print(f"📥 politicas_regulaciones : {df_reg.shape}")
print(f"📥 politicas_acuerdos     : {df_acu.shape}")

# ═══════════════════════════════════════════════════════════════════════
# TABLA 1 — tiene_subsidio_activo (pais × anio → flag 0/1)
# Decisión arquitecto JOIN-2 Opción B: lookup binario
# ═══════════════════════════════════════════════════════════════════════
# Expandir cada subsidio a todos los años del proyecto
ANIOS_PROYECTO = list(range(2011, 2034))

subsidio_rows = []
for _, row in df_sub.iterrows():
    for anio in ANIOS_PROYECTO:
        if anio >= row['anio']:   # subsidio vigente desde su año de concesión
            subsidio_rows.append({
                'pais': row['nombre_pais'],
                'anio': anio,
                'tiene_subsidio_activo': 1
            })

subsidio_long = pd.DataFrame(subsidio_rows)
tabla_subsidios = (
    subsidio_long
    .groupby(['pais', 'anio'], as_index=False)
    .agg(tiene_subsidio_activo=('tiene_subsidio_activo', 'max'))
)

print(f"\n{'='*52}")
print(f"  TABLA 1 — tabla_subsidios (pais × anio)")
print(f"{'='*52}")
print(f"  Shape: {tabla_subsidios.shape}")
print(f"  Países: {sorted(tabla_subsidios['pais'].unique())}")
print(f"  % filas con subsidio activo: "
      f"{tabla_subsidios['tiene_subsidio_activo'].mean()*100:.1f}%")

# ═══════════════════════════════════════════════════════════════════════
# TABLA 2 — tabla_regulaciones (pais × anio → flags binarios por tipo)
# Decisión arquitecto JOIN-4: estático + normativa_vigente temporal
# ═══════════════════════════════════════════════════════════════════════
tipos_regulacion = df_reg['tipo'].unique()
print(f"\n  Tipos de regulación: {sorted(tipos_regulacion)}")

reg_rows = []
for _, row in df_reg.iterrows():
    anio_impl = row['anio_implementacion']
    for anio in ANIOS_PROYECTO:
        vigente = 1 if (pd.notna(anio_impl) and anio >= int(anio_impl)) else 0
        reg_rows.append({
            'pais'          : row['nombre_pais'],
            'anio'          : anio,
            'tipo_regulacion': row['tipo'],
            'normativa_vigente': vigente
        })

reg_long = pd.DataFrame(reg_rows)

# Pivot: una columna flag por tipo de regulación
tabla_regulaciones = (
    reg_long[reg_long['normativa_vigente'] == 1]
    .groupby(['pais', 'anio', 'tipo_regulacion'])
    .size().reset_index(name='v')
    .pivot_table(index=['pais','anio'], columns='tipo_regulacion',
                 values='v', fill_value=0)
    .clip(upper=1)   # binarizar (puede haber 2 regulaciones del mismo tipo)
    .reset_index()
)
# Renombrar columnas para claridad
tabla_regulaciones.columns.name = None
tabla_regulaciones.columns = (
    ['pais', 'anio'] +
    [f'reg_{c}' for c in tabla_regulaciones.columns[2:]]
)

print(f"\n{'='*52}")
print(f"  TABLA 2 — tabla_regulaciones (pais × anio)")
print(f"{'='*52}")
print(f"  Shape: {tabla_regulaciones.shape}")
print(f"  Columnas: {list(tabla_regulaciones.columns)}")

# ═══════════════════════════════════════════════════════════════════════
# TABLA 3 — tabla_acuerdos (pais → lookup estático)
# Decisión arquitecto JOIN-4: contexto estático por país
# ═══════════════════════════════════════════════════════════════════════

# 1. Asegurar que los 12 países del proyecto aparecen (aunque sin acuerdos)
PAISES_PROYECTO = [
    'Alemania', 'Argentina', 'Australia', 'Brasil', 'China',
    'España', 'Estados Unidos', 'Francia', 'India',
    'Kenia', 'México', 'Nueva Zelanda'
]

acuerdos_pivot = (
    df_acu[df_acu['firmado'] == True]
    .groupby(['nombre_pais', 'nombre_acuerdo'])
    .size().reset_index(name='v')
    .pivot_table(index='nombre_pais', columns='nombre_acuerdo',
                 values='v', fill_value=0)
    .clip(upper=1)
    .reset_index()
)
acuerdos_pivot.columns.name = None
acuerdos_pivot.rename(columns={'nombre_pais': 'pais'}, inplace=True)
acuerdos_pivot.columns = (
    ['pais'] +
    [f'acuerdo_{c.replace(" ","_")}' for c in acuerdos_pivot.columns[1:]]
)

# Añadir objetivo_reduccion_emisiones si existe
if 'objetivo_reduccion_emisiones' in df_acu.columns:
    obj_red = (
        df_acu[df_acu['objetivo_reduccion_emisiones'].notna()]
        .groupby('nombre_pais')['objetivo_reduccion_emisiones']
        .max()
        .reset_index()
        .rename(columns={'nombre_pais': 'pais',
                         'objetivo_reduccion_emisiones': 'objetivo_reduccion_emisiones_pct'})
    )
    acuerdos_pivot = acuerdos_pivot.merge(obj_red, on='pais', how='left')

    # Añadir países faltantes con 0 en todos los acuerdos
paises_faltantes = set(PAISES_PROYECTO) - set(acuerdos_pivot['pais'])
if paises_faltantes:
    print(f"  ⚠️  Países sin acuerdos firmados (se añaden con ceros): "
          f"{sorted(paises_faltantes)}")
    cols_acuerdo = [c for c in acuerdos_pivot.columns if c != 'pais']
    filas_nuevas = pd.DataFrame({
        'pais': list(paises_faltantes),
        **{col: 0 for col in cols_acuerdo}
    })
    acuerdos_pivot = pd.concat(
        [acuerdos_pivot, filas_nuevas], ignore_index=True
    ).sort_values('pais').reset_index(drop=True)

# 2. Imputar NaN en objetivo_reduccion_emisiones_pct → 0
#    (ausencia de objetivo declarado = 0%, no dato desconocido)
acuerdos_pivot['objetivo_reduccion_emisiones_pct'] = (
    acuerdos_pivot['objetivo_reduccion_emisiones_pct'].fillna(0.0)
)

# 3. Convertir todas las columnas de acuerdo a int (están como float por el pivot)
cols_flags = [c for c in acuerdos_pivot.columns
              if c not in ('pais', 'objetivo_reduccion_emisiones_pct')]
acuerdos_pivot[cols_flags] = acuerdos_pivot[cols_flags].astype(int)

# ── Validación final ───────────────────────────────────────────────────────
print(f"\n{'='*52}")
print(f"  tabla_acuerdos — POST PATCH")
print(f"{'='*52}")
print(f"  Shape          : {acuerdos_pivot.shape}")
print(f"  Países         : {sorted(acuerdos_pivot['pais'].unique())}")
print(f"  Nulos          : {acuerdos_pivot.isnull().sum().sum()}")
print(f"  Países faltantes en proyecto : "
      f"{set(PAISES_PROYECTO) - set(acuerdos_pivot['pais'])}")
print(f"\n  ✅ Todas las tablas PRE-D validadas y listas:")
print(f"     · tabla_subsidios    : {tabla_subsidios.shape}   — 12 países ✓")
print(f"     · tabla_regulaciones : {tabla_regulaciones.shape} — flags temporales ✓")
print(f"     · tabla_acuerdos     : {acuerdos_pivot.shape}   — 12 países, 0 nulos ✓")

📥 politicas_subsidios    : (37, 8)
📥 politicas_regulaciones : (29, 8)
📥 politicas_acuerdos     : (22, 7)

  TABLA 1 — tabla_subsidios (pais × anio)
  Shape: (181, 3)
  Países: ['Alemania', 'Argentina', 'Australia', 'Brasil', 'China', 'España', 'Estados Unidos', 'Francia', 'India', 'Kenia', 'México', 'Nueva Zelanda']
  % filas con subsidio activo: 100.0%

  Tipos de regulación: ['bienestar_animal', 'deforestacion', 'emisiones_ganaderas', 'etanol_mezcla', 'pesticidas_neonicotinoides', 'reservas_legales', 'rotacion_cultivos', 'uso_agua']

  TABLA 2 — tabla_regulaciones (pais × anio)
  Shape: (236, 10)
  Columnas: ['pais', 'anio', 'reg_bienestar_animal', 'reg_deforestacion', 'reg_emisiones_ganaderas', 'reg_etanol_mezcla', 'reg_pesticidas_neonicotinoides', 'reg_reservas_legales', 'reg_rotacion_cultivos', 'reg_uso_agua']
  ⚠️  Países sin acuerdos firmados (se añaden con ceros): ['Argentina', 'Nueva Zelanda']

  tabla_acuerdos — POST PATCH
  Shape          : (12, 8)
  Países         : ['Alema

## Sección 6 — Decisiones de Integración: Justificación

### Contexto

Antes de construir los datasets maestros, realizamos un análisis sobre los resultados obtenidos en este bloque PRE.
Este análisis reveló gaps estructurales que condicionan qué joins son
estadísticamente válidos para entrenar modelos de Machine Learning.

---

### Gaps detectados

| Cruce | Solapamiento temporal | Cobertura países | Decisión |
|---|---|---|---|
| Producción Agrícola ↔ Clima | 100% (2011–2020) | 10/12 países | ✅ **JOIN** |
| Producción Ganadera ↔ Clima | 100% (2011–2020) | 10/12 países | ✅ **JOIN** |
| Producción Agrícola ↔ Políticas | Estático/lookup | 12/12 países | ✅ **JOIN** |
| Producción Agrícola ↔ Plagas | 1 año (solo 2020) | 8/12 países | ❌ **DESCARTADO** |
| Precios ↔ Noticias/Sentimiento | 2021–2023 | 22.9% cobertura | ❌ **DESCARTADO** |
| Imágenes ↔ Clima | Diferente propósito | — | ❌ **DESCARTADO** |

---

### Datasets maestros a construir

Únicamente se construirán joins que garanticen integridad estadística real:

**`master_agricola.csv`**
Integra `produccion_agricola` + `condiciones_climaticas` + `politicas` (lookup).
Solapamiento temporal del 100%. NaN estructurales mínimos y justificados
(España y Nueva Zelanda sin datos climáticos).
Alimenta los problemas **P1** (Predicción de Rendimiento), **P5** (Detección
de Anomalías) y **P6** (Recomendación de Cultivos).

**`master_ganadero.csv`**
Integra `produccion_ganadera` + `condiciones_climaticas` + `politicas` (lookup).
Misma lógica que el anterior. Alimenta el modelo **P3** (Segmentación de Países).

---

### Datasets a utilizar de forma individual

Los siguientes datasets se usan tal cual, sin joins adicionales, porque
unirlos introduciría más ruido que señales:

- **`precios_mercado_processed.csv`** → P4 (Serie temporal pura). El cruce
  con noticias solo cubría el 22.9% de las combinaciones país × mes, por lo
  que el 77% restante se habría imputado a neutro (0), destruyendo la
  variabilidad necesaria para el modelo predictivo.

- **`reportes_plagas_processed.csv`** → P2 (Clasificación de riesgo).
  Solo 1 año de solapamiento con producción agrícola y 8 de 12 países con
  cobertura. Sus propias features internas (severidad, área afectada,
  eficacia del tratamiento) son suficientes para entrenar el clasificador.

- **`imagenes_metadata_processed.csv`** → P5 (Visión por computador).
  Sin los píxeles reales, cruzar metadatos con clima aportaría valor
  analítico ínfimo. Se reserva como dataset auxiliar exploratorio.

---

### Conclusión

Esta decisión prioriza la **calidad del dato sobre la cantidad de features**.
Un modelo entrenado sobre datos bien integrados y con NaN justificados
superará siempre a uno con numerosas columnas producto de joins forzados.
Los gaps temporales y de cobertura han sido detectados, documentados y
gestionados de forma explícita — lo que constituye en sí mismo un resultado
analítico relevante del proyecto.

## 4.1. master_agricola.csv

**Datasets que se integran:**

| Dataset | Granularidad | Período | Países |
|---|---|---|---|
| `produccion_agricola_processed` | pais × cultivo × anio | 2011–2020 | 12 |
| `clima_pais_anio` [PRE-A] | pais × anio | 2011–2020 | 10 |
| `tabla_subsidios` [PRE-D] | pais × anio | 2011–2033 | 12 |
| `tabla_regulaciones` [PRE-D] | pais × anio | variable | 12 |
| `tabla_acuerdos` [PRE-D] | pais (estático) | — | 12 |

**Estrategia de join:** LEFT JOIN desde `produccion_agricola` como tabla maestra.
Todos los enriquecimientos son opcionales — un NaN en clima o regulaciones
no invalida la fila de producción.

**Gaps conocidos y tratamiento:**
- **Clima:** España y Nueva Zelanda no tienen datos → NaN aceptado.
  Se creará `flag_sin_clima` (0/1) para que el modelo distinga ausencia real.
- **Regulaciones:** Solo se unen los años donde la normativa ya está vigente.
  Años anteriores a `anio_implementacion` → 0 (no vigente), no NaN.
- **Subsidios:** Cobertura 100% desde el año de concesión → 0 NaN esperados.

**Modelos que alimenta:**
- **P1** — Predicción de Rendimiento Agrícola (ton/ha): target = `rendimiento_ton_ha`
- **P5** — Detección de Anomalías en Producción: target = outliers en producción/rendimiento
- **P6** — Recomendación de Cultivos: features de clima + políticas por país

### Celda 1 - Nueva característica: precio histórico por cultivo

In [5]:
# Dado el gap temporal (producción 2011-2020, precios 2021+),
# no es posible un join directo por año.
# → Creamos "Contexto Económico Histórico": precio medio por pais × cultivo
# como señal de si ese cultivo era valioso en el mercado histórico,
# lo que pudo incentivar mejores técnicas y mayor rendimiento.

df_precios = pd.read_csv('../../data/processed/precios_mercado_processed.csv')
print(f"📥 precios_mercado_processed: {df_precios.shape}")
print(f"   Productos en precios: {sorted(df_precios['producto'].unique())}")
print(f"   Cultivos en produccion: "
      f"{sorted(pd.read_csv('../../data/processed/produccion_agricola_processed.csv')['cultivo'].unique())}")

# ── Precio histórico medio por pais × cultivo ─────────────────────────────
precio_historico = (
    df_precios
    .groupby(['pais', 'producto'], as_index=False)
    .agg(
        precio_hist_medio_usd   = ('precio_usd_ton', 'mean'),
        precio_hist_max_usd     = ('precio_usd_ton', 'max'),
        precio_hist_volatilidad = ('precio_usd_ton', 'std'),
    )
)
precio_historico['precio_hist_medio_usd']   = precio_historico['precio_hist_medio_usd'].round(2)
precio_historico['precio_hist_max_usd']     = precio_historico['precio_hist_max_usd'].round(2)
precio_historico['precio_hist_volatilidad'] = precio_historico['precio_hist_volatilidad'].round(2)

# ── Verificar solapamiento de cultivos con produccion_agricola ─────────────
df_prod = pd.read_csv('../../data/processed/produccion_agricola_processed.csv')
productos_prod    = set(df_prod['cultivo'].unique())
productos_precios = set(precio_historico['producto'].unique())

solo_en_prod    = productos_prod - productos_precios
solo_en_precios = productos_precios - productos_prod
en_ambos        = productos_prod & productos_precios

print(f"\n{'='*52}")
print(f"  Solapamiento de productos pais × producto")
print(f"{'='*52}")
print(f"  Productos en producción     : {len(productos_prod)}")
print(f"  Productos en precios        : {len(productos_precios)}")
print(f"  En ambos ✅                : {len(en_ambos)} → {sorted(en_ambos)}")
print(f"  Solo en producción ⚠️      : {sorted(solo_en_prod)}")
print(f"  Solo en precios (se pierde): {sorted(solo_en_precios)}")

paises_prod    = set(df_prod['pais'].unique())
paises_precios = set(precio_historico['pais'].unique())
print(f"\n  Países solo en producción  : {paises_prod - paises_precios}")
print(f"  Países solo en precios     : {paises_precios - paises_prod}")

print(f"\n  Vista previa precio_historico:")
display(precio_historico.head(6))
print(f"\n  ✅ precio_historico lista en memoria: {precio_historico.shape}")

📥 precios_mercado_processed: (864, 15)
   Productos en precios: ['Arroz', 'Carne_bovina', 'Carne_porcina', 'Leche', 'Maíz', 'Soja', 'Trigo']
   Cultivos en produccion: ['Algodón', 'Arroz', 'Café', 'Caña de azúcar', 'Cebada', 'Girasol', 'Maíz', 'Soja', 'Trigo', 'Té']

  Solapamiento de productos pais × producto
  Productos en producción     : 10
  Productos en precios        : 7
  En ambos ✅                : 4 → ['Arroz', 'Maíz', 'Soja', 'Trigo']
  Solo en producción ⚠️      : ['Algodón', 'Café', 'Caña de azúcar', 'Cebada', 'Girasol', 'Té']
  Solo en precios (se pierde): ['Carne_bovina', 'Carne_porcina', 'Leche']

  Países solo en producción  : {'Canadá', 'Italia', 'Rusia'}
  Países solo en precios     : set()

  Vista previa precio_historico:


,pais,producto,precio_hist_medio_usd,precio_hist_max_usd,precio_hist_volatilidad
0,Alemania,Arroz,361.64,430.07,35.23
1,Alemania,Carne_bovina,4103.42,4672.01,400.50
2,Alemania,Carne_porcina,2460.32,2779.00,192.65
3,Alemania,Leche,348.64,374.53,16.82
4,Alemania,Maíz,202.71,242.78,25.59
5,Alemania,Soja,513.15,632.26,53.02



  ✅ precio_historico lista en memoria: (84, 5)


### Celda 2 - JOIN 1 - Ensamblaje de master_agricola

In [6]:
df_prod = pd.read_csv('../../data/processed/produccion_agricola_processed.csv')
print(f"📥 produccion_agricola_processed: {df_prod.shape}")
print(f"   Columnas: {list(df_prod.columns)}")

# ── Preparar precio_historico con clave 'cultivo' para el join ─────────────
precio_historico_join = precio_historico.rename(columns={'producto': 'cultivo'})

# ── JOIN 1/5: produccion × clima (pais × anio) ────────────────────────────
master = df_prod.merge(
    clima_pais_anio,
    on=['pais', 'anio'],
    how='left'
)
print(f"\n  Tras JOIN clima        : {master.shape} | "
      f"NaN clima: {master[['temperatura_promedio']].isnull().sum().values[0]}")

# ── JOIN 2/5: × subsidios (pais × anio → flag 0/1) ────────────────────────
master = master.merge(
    tabla_subsidios,
    on=['pais', 'anio'],
    how='left'
)
master['tiene_subsidio_activo'] = (
    master['tiene_subsidio_activo'].fillna(0).astype(int)
)
print(f"  Tras JOIN subsidios    : {master.shape} | "
      f"NaN subsidio: {master['tiene_subsidio_activo'].isnull().sum()}")

# ── JOIN 3/5: × regulaciones (pais × anio → flags reg_*) ──────────────────
master = master.merge(
    tabla_regulaciones,
    on=['pais', 'anio'],
    how='left'
)
cols_reg = [c for c in master.columns if c.startswith('reg_')]
for col in cols_reg:
    master[col] = master[col].fillna(0).astype(int)
print(f"  Tras JOIN regulaciones : {master.shape} | "
      f"NaN reg: {master[cols_reg].isnull().sum().sum()}")

# ── JOIN 4/5: × acuerdos (pais → lookup estático) ─────────────────────────
master = master.merge(
    acuerdos_pivot,
    on=['pais'],
    how='left'
)
cols_acu = [c for c in master.columns if c.startswith('acuerdo_')]
for col in cols_acu:
    master[col] = master[col].fillna(0)
print(f"  Tras JOIN acuerdos     : {master.shape} | "
      f"NaN acuerdos: {master[cols_acu].isnull().sum().sum()}")

# ── JOIN 5/5: × precio histórico (pais × cultivo) ─────────────────────────
master = master.merge(
    precio_historico_join,
    on=['pais', 'cultivo'],
    how='left'
)
cols_precio = ['precio_hist_medio_usd', 'precio_hist_max_usd',
               'precio_hist_volatilidad']
n_nan_precio = master['precio_hist_medio_usd'].isnull().sum()
print(f"  Tras JOIN precio hist  : {master.shape} | "
      f"NaN precio: {n_nan_precio} "
      f"({n_nan_precio/len(master)*100:.1f}% — cultivos sin precio histórico)")

# ── Flag de cobertura climática ────────────────────────────────────────────
master['flag_sin_clima'] = master['temperatura_promedio'].isnull().astype(int)

# ── Verificación de shape esperada ────────────────────────────────────────
print(f"\n{'='*52}")
print(f"  master_agricola — RESULTADO")
print(f"{'='*52}")
print(f"  Shape                   : {master.shape}")
print(f"  Filas esperadas (prod)  : {len(df_prod)}")
print(f"  Duplicación de filas    : {'⚠️ SÍ' if len(master) > len(df_prod) else '✅ NO'}")
print(f"\n  NaN por columna (solo columnas con NaN):")
nan_cols = master.isnull().sum()
nan_cols = nan_cols[nan_cols > 0]
for col, n in nan_cols.items():
    print(f"    {col:<40} : {n} ({n/len(master)*100:.1f}%)")
print(f"\n  Países en master: {sorted(master['pais'].unique())}")
print(f"  Cultivos en master: {sorted(master['cultivo'].unique())}")
print(f"  Rango temporal: {master['anio'].min()} – {master['anio'].max()}")
print(f"\n  Vista previa:")
display(master.head(3))

📥 produccion_agricola_processed: (1200, 11)
   Columnas: ['pais', 'codigo_iso', 'region', 'cultivo', 'anio', 'superficie_hectareas', 'rendimiento_ton_ha', 'produccion_ton', 'fertilizantes_kg_ha', 'agua_riego_m3_ha', 'tendencia_5_anios']

  Tras JOIN clima        : (1200, 31) | NaN clima: 400
  Tras JOIN subsidios    : (1200, 32) | NaN subsidio: 0
  Tras JOIN regulaciones : (1200, 40) | NaN reg: 0
  Tras JOIN acuerdos     : (1200, 47) | NaN acuerdos: 0
  Tras JOIN precio hist  : (1200, 50) | NaN precio: 850 (70.8% — cultivos sin precio histórico)

  master_agricola — RESULTADO
  Shape                   : (1200, 51)
  Filas esperadas (prod)  : 1200
  Duplicación de filas    : ✅ NO

  NaN por columna (solo columnas con NaN):
    codigo_iso_y                             : 400 (33.3%)
    temperatura_promedio                     : 400 (33.3%)
    temperatura_maxima                       : 400 (33.3%)
    temperatura_minima                       : 400 (33.3%)
    precipitacion_total         

,pais,codigo_iso_x,region,cultivo,anio,superficie_hectareas,rendimiento_ton_ha,produccion_ton,fertilizantes_kg_ha,agua_riego_m3_ha,...,acuerdo_Acuerdo_Paris,acuerdo_Convenio_Biodiversidad,acuerdo_ODS_2030,acuerdo_Protocolo_Kyoto,acuerdo_TLCAN,objetivo_reduccion_emisiones_pct,precio_hist_medio_usd,precio_hist_max_usd,precio_hist_volatilidad,flag_sin_clima
0,Argentina,ARG,América del Sur,Soja,2011,4703643,2.22,10465461,105,4811,...,0.0,0.0,0.0,0.0,0.0,0.0,502.72,627.89,61.57,0
1,Argentina,ARG,América del Sur,Maíz,2011,3371781,8.45,28485921,164,5557,...,0.0,0.0,0.0,0.0,0.0,0.0,201.47,225.88,18.42,0
2,Argentina,ARG,América del Sur,Cebada,2011,1661577,2.77,4596917,105,6514,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0


Problema 1 — precio_historico: 70.8% NaN ⚠️
Tienes razón en cuestionarlo. Con solo 4 de 10 cultivos cubiertos (Arroz, Maíz, Soja, Trigo), esta feature tiene baja utilidad directa. Decisión recomendada: mantenerla pero como feature auxiliar, no como feature principal. En Fase 3 (Feature Engineering) se imputará por mediana de cultivo y el modelo de importancia de variables (SHAP) dirá si aporta algo real. El coste de mantenerla es nulo; el coste de eliminarla ahora y luego querer recuperarla es mayor.

Problema 2 — clima: 33.3% NaN 🚨 (más grave de lo esperado)
Esperábamos ~16% (España + Nueva Zelanda = 2 países). Tenemos 33.3% = 4 países sin clima. Esto significa que produccion_agricola tiene 15 países, no 12 — incluye Canadá, Rusia e Italia, que no estaban en condiciones_climaticas. Hay que verificarlo.

### Celda 3 - Validación y limpieza post-join

In [7]:
# ── Diagnóstico 1: países reales en master ────────────────────────────────
paises_master  = set(master['pais'].unique())
paises_clima   = set(clima_pais_anio['pais'].unique())
paises_sin_clima = paises_master - paises_clima

print(f"  Países en master          : {len(paises_master)} → {sorted(paises_master)}")
print(f"  Países con clima          : {len(paises_clima)}")
print(f"  Países SIN clima ⚠️       : {sorted(paises_sin_clima)}")

filas_sin_clima = master[master['flag_sin_clima'] == 1]
print(f"\n  Filas sin clima           : {len(filas_sin_clima)} "
      f"({len(filas_sin_clima)/len(master)*100:.1f}%)")
print(f"  Distribución por país:")
print(filas_sin_clima['pais'].value_counts().to_string())

# ── Diagnóstico 2: columna codigo_iso duplicada ───────────────────────────
# El join con clima generó codigo_iso_x y codigo_iso_y → limpiar
print(f"\n  Columnas duplicadas detectadas: "
      f"{[c for c in master.columns if c.endswith('_x') or c.endswith('_y')]}")

master = master.drop(columns=['codigo_iso_y'], errors='ignore')
master = master.rename(columns={'codigo_iso_x': 'codigo_iso'})
print(f"  ✅ codigo_iso_y eliminada, codigo_iso_x renombrada a codigo_iso")

# ── Diagnóstico 3: precio_historico — decisión de tratamiento ─────────────
n_nan_precio = master['precio_hist_medio_usd'].isnull().sum()
print(f"\n  NaN en precio_hist_medio_usd : {n_nan_precio} ({n_nan_precio/len(master)*100:.1f}%)")
print(f"  Cultivos SIN precio histórico: "
      f"{sorted(set(master[master['precio_hist_medio_usd'].isna()]['cultivo'].unique()))}")
print(f"  Cultivos CON precio histórico: "
      f"{sorted(set(master[master['precio_hist_medio_usd'].notna()]['cultivo'].unique()))}")

# Decisión: mantener feature pero flag explícito
master['flag_sin_precio_historico'] = (
    master['precio_hist_medio_usd'].isnull().astype(int)
)
# NO imputar aquí — se hará en Fase 3 (Feature Engineering)
# Documentar que NaN = cultivo sin mercado global de referencia en el dataset

# ── Diagnóstico 4: objetivo_reduccion_emisiones_pct — 20% NaN ─────────────
# Solo México tiene valor (24.0). Imputar a 0 = sin objetivo declarado
master['objetivo_reduccion_emisiones_pct'] = (
    master['objetivo_reduccion_emisiones_pct'].fillna(0.0)
)
print(f"\n  objetivo_reduccion_emisiones_pct NaN tras imputación: "
      f"{master['objetivo_reduccion_emisiones_pct'].isnull().sum()}")

# ── Resumen final de NaN ───────────────────────────────────────────────────
print(f"\n{'='*52}")
print(f"  RESUMEN NaN — master_agricola POST-LIMPIEZA")
print(f"{'='*52}")
nan_resumen = master.isnull().sum()
nan_resumen = nan_resumen[nan_resumen > 0]
if len(nan_resumen) == 0:
    print(f"  ✅ 0 columnas con NaN no justificados")
else:
    print(f"  Columnas con NaN restantes (todos justificados):")
    for col, n in nan_resumen.items():
        pct = n / len(master) * 100
        if col.startswith(('temperatura', 'precipi', 'humedad', 'dias_',
                            'meses_', 'indice_', 'n_eventos', 'ev_',
                            'flag_anio', 'precio_hist')):
            motivo = "sin datos climáticos" if not col.startswith('precio') \
                     else "cultivo sin precio de mercado"
        else:
            motivo = "revisar"
        print(f"    {col:<42} : {n:>4} ({pct:.1f}%) — {motivo}")

print(f"\n  Shape final: {master.shape}")
print(f"  ✅ master listo para exportar")

  Países en master          : 15 → ['Alemania', 'Argentina', 'Australia', 'Brasil', 'Canadá', 'China', 'España', 'Estados Unidos', 'Francia', 'India', 'Italia', 'Kenia', 'México', 'Nueva Zelanda', 'Rusia']
  Países con clima          : 10
  Países SIN clima ⚠️       : ['Canadá', 'España', 'Italia', 'Nueva Zelanda', 'Rusia']

  Filas sin clima           : 400 (33.3%)
  Distribución por país:
pais
Nueva Zelanda    80
España           80
Italia           80
Canadá           80
Rusia            80

  Columnas duplicadas detectadas: ['codigo_iso_x', 'codigo_iso_y']
  ✅ codigo_iso_y eliminada, codigo_iso_x renombrada a codigo_iso

  NaN en precio_hist_medio_usd : 850 (70.8%)
  Cultivos SIN precio histórico: ['Algodón', 'Arroz', 'Café', 'Caña de azúcar', 'Cebada', 'Girasol', 'Maíz', 'Soja', 'Trigo', 'Té']
  Cultivos CON precio histórico: ['Arroz', 'Maíz', 'Soja', 'Trigo']

  objetivo_reduccion_emisiones_pct NaN tras imputación: 0

  RESUMEN NaN — master_agricola POST-LIMPIEZA
  Columnas con N

- Los 5 países sin `clima` están identificados — Canadá, España, Italia, Nueva Zelanda y Rusia simplemente no existían en condiciones_climaticas. Son 5 × 10 años × 8 cultivos = 400 filas exactas. Correcto.

- `precio_hist` — el diagnóstico muestra curiosamente que todos los cultivos aparecen en "CON precio histórico" — eso significa que todos los cultivos tienen al menos alguna fila con precio, pero los 850 NaN son las combinaciones pais × cultivo donde ese cultivo existe en producción pero no en precios. Coherente.

- 0 NaN no justificados — el master está listo para exportar.

### Celda 4 - Exportacion master_agricola.csv

In [ ]:
import os

# ── Orden final de columnas ────────────────────────────────────────────────
# Agrupar por bloque temático para legibilidad en EDA y PowerBI
cols_id         = ['pais', 'codigo_iso', 'region', 'cultivo', 'anio']
cols_produccion = ['superficie_hectareas', 'rendimiento_ton_ha',
                   'produccion_ton', 'fertilizantes_kg_ha',
                   'agua_riego_m3_ha', 'tendencia_5_anios']
cols_clima      = ['temperatura_promedio', 'temperatura_maxima',
                   'temperatura_minima', 'precipitacion_total',
                   'humedad_relativa_promedio', 'dias_con_heladas',
                   'meses_estres_hidrico', 'indice_aridez']
cols_eventos    = ['n_eventos_total_regiones', 'ev_granizo', 'ev_helada_tardia',
                   'ev_incendios', 'ev_inundacion', 'ev_ola_calor',
                   'ev_sequia_extrema', 'ev_sequia_moderada',
                   'ev_sequia_severa', 'ev_tormenta_severa',
                   'flag_anio_eventos_extremos']
cols_politicas  = ['tiene_subsidio_activo',
                   'reg_bienestar_animal', 'reg_deforestacion',
                   'reg_emisiones_ganaderas', 'reg_etanol_mezcla',
                   'reg_pesticidas_neonicotinoides', 'reg_reservas_legales',
                   'reg_rotacion_cultivos', 'reg_uso_agua',
                   'acuerdo_Acuerdo_Mercosur', 'acuerdo_Acuerdo_Paris',
                   'acuerdo_Convenio_Biodiversidad', 'acuerdo_ODS_2030',
                   'acuerdo_Protocolo_Kyoto', 'acuerdo_TLCAN',
                   'objetivo_reduccion_emisiones_pct']
cols_precio     = ['precio_hist_medio_usd', 'precio_hist_max_usd',
                   'precio_hist_volatilidad']
cols_flags      = ['flag_sin_clima', 'flag_sin_precio_historico']

# Verificar que no se pierde ninguna columna en el reordenado
cols_ordenadas = (cols_id + cols_produccion + cols_clima +
                  cols_eventos + cols_politicas + cols_precio + cols_flags)
cols_faltantes = set(master.columns) - set(cols_ordenadas)
if cols_faltantes:
    print(f"  ⚠️  Columnas no asignadas a ningún bloque: {cols_faltantes}")
    cols_ordenadas += list(cols_faltantes)  # añadir al final para no perderlas

master = master[cols_ordenadas]

# ── Exportación ────────────────────────────────────────────────────────────
output_path = '../../data/processed/master_agricola.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
master.to_csv(output_path, index=False, encoding='utf-8-sig')

# ── Verificación post-exportación ─────────────────────────────────────────
df_check = pd.read_csv(output_path)
print(f"{'='*52}")
print(f"  master_agricola.csv — EXPORTADO")
print(f"{'='*52}")
print(f"  Ruta          : {output_path}")
print(f"  Shape         : {df_check.shape}")
print(f"  Tamaño en disco: {os.path.getsize(output_path)/1024:.1f} KB")
print(f"  Columnas ({len(df_check.columns)}):")
for bloque, cols in [
    ("🔑 Identificadores", cols_id),
    ("🌾 Producción",      cols_produccion),
    ("🌡️  Clima",           cols_clima),
    ("⛈️  Eventos extremos", cols_eventos),
    ("📋 Políticas",       cols_politicas),
    ("💰 Precio histórico", cols_precio),
    ("🚩 Flags de calidad", cols_flags),
]:
    print(f"\n  {bloque}:")
    print(f"    {[c for c in cols if c in df_check.columns]}")

print(f"\n  NaN por bloque:")
for bloque, cols in [
    ("Clima",           cols_clima),
    ("Precio histórico", cols_precio),
]:
    n = df_check[[c for c in cols if c in df_check.columns]].isnull().sum().sum()
    print(f"    {bloque:<20}: {n} NaN ({n/(len(df_check)*len(cols))*100:.1f}% del bloque)")

print(f"\n  ✅ master_agricola.csv listo para EDA y modelado (P1, P5, P6)")

  master_agricola.csv — EXPORTADO
  Ruta          : ../../data/processed/master_agricola.csv
  Shape         : (1200, 51)
  Tamaño en disco: 245.0 KB
  Columnas (51):

  🔑 Identificadores:
    ['pais', 'codigo_iso', 'region', 'cultivo', 'anio']

  🌾 Producción:
    ['superficie_hectareas', 'rendimiento_ton_ha', 'produccion_ton', 'fertilizantes_kg_ha', 'agua_riego_m3_ha', 'tendencia_5_anios']

  🌡️  Clima:
    ['temperatura_promedio', 'temperatura_maxima', 'temperatura_minima', 'precipitacion_total', 'humedad_relativa_promedio', 'dias_con_heladas', 'meses_estres_hidrico', 'indice_aridez']

  ⛈️  Eventos extremos:
    ['n_eventos_total_regiones', 'ev_granizo', 'ev_helada_tardia', 'ev_incendios', 'ev_inundacion', 'ev_ola_calor', 'ev_sequia_extrema', 'ev_sequia_moderada', 'ev_sequia_severa', 'ev_tormenta_severa', 'flag_anio_eventos_extremos']

  📋 Políticas:
    ['tiene_subsidio_activo', 'reg_bienestar_animal', 'reg_deforestacion', 'reg_emisiones_ganaderas', 'reg_etanol_mezcla', 'reg_pesti

## 4.2. master_ganadero.csv

**Datasets que se integran:**

| Dataset | Granularidad | Período | Países |
|---|---|---|---|
| `produccion_ganadera_processed` | pais × especie × anio | 2011–2020 | 12 |
| `clima_pais_anio` [PRE-A] | pais × anio | 2011–2020 | 10 |
| `tabla_subsidios` [PRE-D] | pais × anio | 2011–2033 | 12 |
| `tabla_regulaciones` [PRE-D] | pais × anio | variable | 12 |
| `tabla_acuerdos` [PRE-D] | pais (estático) | — | 12 |

**Diferencias respecto a master_agricola:**
- La unidad de análisis es **especie ganadera**, no cultivo
- Las regulaciones más relevantes aquí son `reg_bienestar_animal`
  y `reg_emisiones_ganaderas` — directamente relacionadas con ganadería
- **No hay feature de precio histórico** — `precios_mercado` incluye
  productos ganaderos (Carne_bovina, Carne_porcina, Leche) pero la
  decisión arquitectónica descartó ese join (Tier 2 standalone)

**Gaps conocidos (idénticos a master_agricola):**
- Clima: Canadá, España, Italia, Nueva Zelanda, Rusia → NaN + `flag_sin_clima`
- Subsidios, regulaciones, acuerdos: cobertura 100% → 0 NaN esperados

**Modelos que alimenta:**
- **P3** — Segmentación de Países por Patrones Productivos (clustering)

### Celda 1 - JOIN 2 - Ensamblaje de master_ganadero.csv

In [11]:
df_gan = pd.read_csv('../../data/processed/produccion_ganadera_processed.csv')
print(f"📥 produccion_ganadera_processed: {df_gan.shape}")
print(f"   Columnas  : {list(df_gan.columns)}")
print(f"   Tipo ganado  : {sorted(df_gan['tipo_ganado'].unique())}")
print(f"   Países    : {sorted(df_gan['pais'].unique())}")
print(f"   Rango     : {df_gan['anio'].min()} – {df_gan['anio'].max()}")

# ── JOIN 1/4: produccion_ganadera × clima (pais × anio) ───────────────────
master_g = df_gan.merge(
    clima_pais_anio,
    on=['pais', 'anio'],
    how='left'
)
print(f"\n  Tras JOIN clima        : {master_g.shape} | "
      f"NaN clima: {master_g['temperatura_promedio'].isnull().sum()}")

# ── JOIN 2/4: × subsidios (pais × anio → flag 0/1) ────────────────────────
master_g = master_g.merge(
    tabla_subsidios,
    on=['pais', 'anio'],
    how='left'
)
master_g['tiene_subsidio_activo'] = (
    master_g['tiene_subsidio_activo'].fillna(0).astype(int)
)
print(f"  Tras JOIN subsidios    : {master_g.shape} | "
      f"NaN subsidio: {master_g['tiene_subsidio_activo'].isnull().sum()}")

# ── JOIN 3/4: × regulaciones (pais × anio → flags reg_*) ──────────────────
master_g = master_g.merge(
    tabla_regulaciones,
    on=['pais', 'anio'],
    how='left'
)
cols_reg = [c for c in master_g.columns if c.startswith('reg_')]
for col in cols_reg:
    master_g[col] = master_g[col].fillna(0).astype(int)
print(f"  Tras JOIN regulaciones : {master_g.shape} | "
      f"NaN reg: {master_g[cols_reg].isnull().sum().sum()}")

# ── JOIN 4/4: × acuerdos (pais → lookup estático) ─────────────────────────
master_g = master_g.merge(
    acuerdos_pivot,
    on=['pais'],
    how='left'
)
cols_acu = [c for c in master_g.columns if c.startswith('acuerdo_')]
for col in cols_acu:
    master_g[col] = master_g[col].fillna(0)
master_g['objetivo_reduccion_emisiones_pct'] = (
    master_g['objetivo_reduccion_emisiones_pct'].fillna(0.0)
)
print(f"  Tras JOIN acuerdos     : {master_g.shape} | "
      f"NaN acuerdos: {master_g[cols_acu].isnull().sum().sum()}")

# ── Limpiar columna duplicada codigo_iso ──────────────────────────────────
if 'codigo_iso_y' in master_g.columns:
    master_g = master_g.drop(columns=['codigo_iso_y'])
    master_g = master_g.rename(columns={'codigo_iso_x': 'codigo_iso'})

# ── Flag de cobertura climática ────────────────────────────────────────────
master_g['flag_sin_clima'] = (
    master_g['temperatura_promedio'].isnull().astype(int)
)

# ── Validación ────────────────────────────────────────────────────────────
print(f"\n{'='*52}")
print(f"  master_ganadero — RESULTADO")
print(f"{'='*52}")
print(f"  Shape                   : {master_g.shape}")
print(f"  Filas esperadas (gan)   : {len(df_gan)}")
print(f"  Duplicación de filas    : "
      f"{'⚠️ SÍ' if len(master_g) > len(df_gan) else '✅ NO'}")

print(f"\n  NaN por columna (solo columnas con NaN):")
nan_cols = master_g.isnull().sum()
nan_cols = nan_cols[nan_cols > 0]
if len(nan_cols) == 0:
    print(f"    ✅ 0 NaN no justificados")
else:
    for col, n in nan_cols.items():
        print(f"    {col:<42} : {n} ({n/len(master_g)*100:.1f}%)")

print(f"\n  Países  : {sorted(master_g['pais'].unique())}")
print(f"  Tipo ganado: {sorted(master_g['tipo_ganado'].unique())}")
print(f"  Rango   : {master_g['anio'].min()} – {master_g['anio'].max()}")
print(f"\n  Vista previa:")
display(master_g.head(3))

📥 produccion_ganadera_processed: (600, 11)
   Columnas  : ['pais', 'anio', 'tipo_ganado', 'cabezas_ganado', 'produccion_carne_ton', 'produccion_leche_lt', 'emisiones_ch4_ton_co2eq', 'intensidad_emisiones', 'eficiencia_carne', 'flag_emision_extrema', 'flag_eficiencia_anomala']
   Tipo ganado  : ['avicola', 'bovino', 'caprino', 'ovino', 'porcino']
   Países    : ['Alemania', 'Argentina', 'Australia', 'Brasil', 'China', 'España', 'Estados Unidos', 'Francia', 'India', 'Kenia', 'México', 'Nueva Zelanda']
   Rango     : 2011 – 2020

  Tras JOIN clima        : (600, 31) | NaN clima: 100
  Tras JOIN subsidios    : (600, 32) | NaN subsidio: 0
  Tras JOIN regulaciones : (600, 40) | NaN reg: 0
  Tras JOIN acuerdos     : (600, 47) | NaN acuerdos: 0

  master_ganadero — RESULTADO
  Shape                   : (600, 48)
  Filas esperadas (gan)   : 600
  Duplicación de filas    : ✅ NO

  NaN por columna (solo columnas con NaN):
    produccion_leche_lt                        : 240 (40.0%)
    codigo_iso

,pais,anio,tipo_ganado,cabezas_ganado,produccion_carne_ton,produccion_leche_lt,emisiones_ch4_ton_co2eq,intensidad_emisiones,eficiencia_carne,flag_emision_extrema,...,reg_rotacion_cultivos,reg_uso_agua,acuerdo_Acuerdo_Mercosur,acuerdo_Acuerdo_Paris,acuerdo_Convenio_Biodiversidad,acuerdo_ODS_2030,acuerdo_Protocolo_Kyoto,acuerdo_TLCAN,objetivo_reduccion_emisiones_pct,flag_sin_clima
0,Argentina,2011,bovino,18160590,1857394,2.355425e+09,1121740,0.062,0.1023,0,...,0,0,0,0,0,0,0,0,0.0,0
1,Argentina,2011,porcino,13481224,1528896,NaN,1984357,0.147,0.1134,0,...,0,0,0,0,0,0,0,0,0.0,0
2,Argentina,2011,avicola,979196737,1290153,NaN,10160819,0.010,0.0013,0,...,0,0,0,0,0,0,0,0,0.0,0


- NaN `clima`: 16.7% (100 filas) — Solo 2 países sin clima esta vez (España y Nueva Zelanda). El ganadero tiene exactamente 12 países, no 15 como el agrícola. Mejor cobertura.

- `produccion_leche_lt`: 40% NaN — Esto NO viene del join, viene del dato original. La leche solo la producen especies bovinas y caprinas — avícola, porcino y ovino no tienen ese dato. Son NaN estructurales por naturaleza del dato, no un problema de integración.

- `codigo_iso`: 100 NaN — El ganadero no tenía `codigo_iso` propio, lo heredó del join con clima. Los 2 países sin clima tampoco tienen código ISO. Hay que gestionarlo.

### Celda 2 - Validación y limpieza post-join

In [ ]:
# ── Diagnóstico 1: produccion_leche_lt — NaN estructural ──────────────────
print(f"  NaN en produccion_leche_lt por tipo de ganado:")
print(master_g.groupby('tipo_ganado')['produccion_leche_lt']
      .apply(lambda x: f"{x.isnull().sum()}/{len(x)} NaN")
      .to_string())
print(f"\n  → NaN estructural: solo bovino/caprino producen leche.")
print(f"    Se añade flag_leche_no_aplica para distinguirlo de dato faltante.")

master_g['flag_leche_no_aplica'] = (
    (~master_g['tipo_ganado'].isin(['bovino', 'caprino'])).astype(int)
)

# ── Diagnóstico 2: codigo_iso heredado del join de clima ──────────────────
# produccion_ganadera no tenía codigo_iso propio.
# El que hay viene de clima_pais_anio → NaN para países sin clima.
# Solución: construir un mapa pais → codigo_iso desde master_agricola
# (que sí tenía codigo_iso original de produccion_agricola)

df_iso_map = (
    pd.read_csv('../../data/processed/produccion_agricola_processed.csv')
    [['pais', 'codigo_iso']]
    .drop_duplicates()
)
paises_sin_iso = set(master_g['pais'].unique()) - set(df_iso_map['pais'].unique())
print(f"\n  Países sin codigo_iso en mapa agrícola: {paises_sin_iso}")

# Reemplazar codigo_iso heredado de clima por el mapa correcto
master_g = master_g.drop(columns=['codigo_iso'], errors='ignore')
master_g = master_g.merge(df_iso_map, on='pais', how='left')
print(f"  NaN en codigo_iso tras reasignación: "
      f"{master_g['codigo_iso'].isnull().sum()}")

# ── Diagnóstico 3: verificar 0 NaN en políticas ───────────────────────────
cols_pol = (['tiene_subsidio_activo'] +
            [c for c in master_g.columns if c.startswith('reg_')] +
            [c for c in master_g.columns if c.startswith('acuerdo_')] +
            ['objetivo_reduccion_emisiones_pct'])
nan_pol = master_g[cols_pol].isnull().sum().sum()
print(f"\n  NaN en bloque políticas: {nan_pol} "
      f"{'✅' if nan_pol == 0 else '⚠️'}")

# ── Resumen final de NaN ───────────────────────────────────────────────────
print(f"\n{'='*52}")
print(f"  RESUMEN NaN — master_ganadero POST-LIMPIEZA")
print(f"{'='*52}")
nan_resumen = master_g.isnull().sum()
nan_resumen = nan_resumen[nan_resumen > 0]
if len(nan_resumen) == 0:
    print(f"  ✅ 0 NaN no justificados")
else:
    motivos = {
        'produccion_leche_lt': 'NaN estructural — hay especies no lácteas (ver flag_leche_no_aplica)',
        'codigo_iso'         : 'país sin código ISO en ningún dataset',
    }
    for col, n in nan_resumen.items():
        pct = n / len(master_g) * 100
        bloque = ('sin datos climáticos' if any(col.startswith(p) for p in
                  ['temperatura','precipi','humedad','dias_','meses_',
                   'indice_','n_eventos','ev_','flag_anio']) else
                  motivos.get(col, 'revisar'))
        print(f"    {col:<42} : {n:>3} ({pct:.1f}%) — {bloque}")

print(f"\n  Shape final: {master_g.shape}") 

  NaN en produccion_leche_lt por tipo de ganado:
tipo_ganado
avicola    120/120 NaN
bovino       0/120 NaN
caprino      0/120 NaN
ovino        0/120 NaN
porcino    120/120 NaN

  → NaN estructural: solo bovino/caprino producen leche.
    Se añade flag_leche_no_aplica para distinguirlo de dato faltante.

  Países sin codigo_iso en mapa agrícola: set()
  NaN en codigo_iso tras reasignación: 0

  NaN en bloque políticas: 0 ✅

  RESUMEN NaN — master_ganadero POST-LIMPIEZA
    produccion_leche_lt                        : 240 (40.0%) — NaN estructural — especie no láctea (ver flag_leche_no_aplica)
    temperatura_promedio                       : 100 (16.7%) — sin datos climáticos
    temperatura_maxima                         : 100 (16.7%) — sin datos climáticos
    temperatura_minima                         : 100 (16.7%) — sin datos climáticos
    precipitacion_total                        : 100 (16.7%) — sin datos climáticos
    humedad_relativa_promedio                  : 100 (16.7%) — si

### Celda 3 - Exportación master_ganadero.csv

In [14]:
# ── Orden final de columnas por bloques temáticos ─────────────────────────
cols_id       = ['pais', 'codigo_iso', 'anio', 'tipo_ganado']
cols_ganadero = ['cabezas_ganado', 'produccion_carne_ton', 'produccion_leche_lt',
                 'emisiones_ch4_ton_co2eq', 'intensidad_emisiones',
                 'eficiencia_carne']
cols_flags_g  = ['flag_emision_extrema', 'flag_eficiencia_anomala',
                 'flag_leche_no_aplica']
cols_clima    = ['temperatura_promedio', 'temperatura_maxima', 'temperatura_minima',
                 'precipitacion_total', 'humedad_relativa_promedio',
                 'dias_con_heladas', 'meses_estres_hidrico', 'indice_aridez']
cols_eventos  = ['n_eventos_total_regiones', 'ev_granizo', 'ev_helada_tardia',
                 'ev_incendios', 'ev_inundacion', 'ev_ola_calor',
                 'ev_sequia_extrema', 'ev_sequia_moderada', 'ev_sequia_severa',
                 'ev_tormenta_severa', 'flag_anio_eventos_extremos']
cols_pol      = ['tiene_subsidio_activo',
                 'reg_bienestar_animal', 'reg_deforestacion',
                 'reg_emisiones_ganaderas', 'reg_etanol_mezcla',
                 'reg_pesticidas_neonicotinoides', 'reg_reservas_legales',
                 'reg_rotacion_cultivos', 'reg_uso_agua',
                 'acuerdo_Acuerdo_Mercosur', 'acuerdo_Acuerdo_Paris',
                 'acuerdo_Convenio_Biodiversidad', 'acuerdo_ODS_2030',
                 'acuerdo_Protocolo_Kyoto', 'acuerdo_TLCAN',
                 'objetivo_reduccion_emisiones_pct']
cols_qf       = ['flag_sin_clima']

cols_ordenadas = (cols_id + cols_ganadero + cols_flags_g +
                  cols_clima + cols_eventos + cols_pol + cols_qf)

# Añadir columnas no asignadas al final
cols_extra = [c for c in master_g.columns if c not in cols_ordenadas]
if cols_extra:
    print(f"  ⚠️ Columnas extra no asignadas: {cols_extra}")
    cols_ordenadas += cols_extra

master_g = master_g[cols_ordenadas]

# ── Exportación ────────────────────────────────────────────────────────────
output_path = '../../data/processed/master_ganadero.csv'
master_g.to_csv(output_path, index=False, encoding='utf-8-sig')

df_check_g = pd.read_csv(output_path)
print(f"{'='*52}")
print(f"  master_ganadero.csv — EXPORTADO")
print(f"{'='*52}")
print(f"  Ruta          : {output_path}")
print(f"  Shape         : {df_check_g.shape}")
print(f"  Tamaño        : {os.path.getsize(output_path)/1024:.1f} KB")
print(f"\n  Columnas ({len(df_check_g.columns)}) por bloque:")
for nombre, cols in [
    ("🔑 Identificadores",    cols_id),
    ("🐄 Ganadería",          cols_ganadero),
    ("🚩 Flags ganadería",    cols_flags_g),
    ("🌡️  Clima",              cols_clima),
    ("⛈️  Eventos extremos",   cols_eventos),
    ("📋 Políticas",          cols_pol),
    ("🚩 Flag calidad join",  cols_qf),
]:
    presentes = [c for c in cols if c in df_check_g.columns]
    print(f"  {nombre}: {presentes}")

print(f"\n  NaN justificados restantes:")
nan_f = df_check_g.isnull().sum()
nan_f = nan_f[nan_f > 0]
for col, n in nan_f.items():
    print(f"    {col:<42}: {n} ({n/len(df_check_g)*100:.1f}%)")

print(f"\n  ✅ master_ganadero.csv listo para EDA y modelado (P3)")

  master_ganadero.csv — EXPORTADO
  Ruta          : ../../data/processed/master_ganadero.csv
  Shape         : (600, 49)
  Tamaño        : 111.6 KB

  Columnas (49) por bloque:
  🔑 Identificadores: ['pais', 'codigo_iso', 'anio', 'tipo_ganado']
  🐄 Ganadería: ['cabezas_ganado', 'produccion_carne_ton', 'produccion_leche_lt', 'emisiones_ch4_ton_co2eq', 'intensidad_emisiones', 'eficiencia_carne']
  🚩 Flags ganadería: ['flag_emision_extrema', 'flag_eficiencia_anomala', 'flag_leche_no_aplica']
  🌡️  Clima: ['temperatura_promedio', 'temperatura_maxima', 'temperatura_minima', 'precipitacion_total', 'humedad_relativa_promedio', 'dias_con_heladas', 'meses_estres_hidrico', 'indice_aridez']
  ⛈️  Eventos extremos: ['n_eventos_total_regiones', 'ev_granizo', 'ev_helada_tardia', 'ev_incendios', 'ev_inundacion', 'ev_ola_calor', 'ev_sequia_extrema', 'ev_sequia_moderada', 'ev_sequia_severa', 'ev_tormenta_severa', 'flag_anio_eventos_extremos']
  📋 Políticas: ['tiene_subsidio_activo', 'reg_bienestar_anima

### Sección 4 — Resumen Ejecutivo: Inventario de Datasets para Modelado

### Datasets Maestros construidos (Tier 1 — Alta Fidelidad)

| Dataset | Filas | Columnas | NaN justificados | Modelos |
|---|---|---|---|---|
| `master_agricola.csv` | 1.200 | 51 | Clima 33% (5 países), Precio 71% (6 cultivos) | P1, P5, P6 |
| `master_ganadero.csv` | 600 | 49 | Clima 17% (2 países), Leche 40% (estructural) | P3 |

### Datasets Standalone (Tier 2 — Uso Directo)

| Dataset | Filas | Columnas | Uso |
|---|---|---|---|
| `precios_mercado_processed.csv` | 864 | 15 | P4 — Serie temporal de precios |
| `reportes_plagas_processed.csv` | 100 | 15 | P2 — Clasificación de riesgo |
| `imagenes_metadata_processed.csv` | 600 | ~20 | P5 — Visión por computador (auxiliar) |

### Datasets Auxiliares (solo EDA y análisis exploratorio)

| Dataset | Uso |
|---|---|
| `noticias_processed.csv` | Correlación sentimiento ↔ precios en EDA |
| `condiciones_climaticas_processed.csv` | Fuente original, no usar directamente en ML |

### Decisiones arquitectónicas que explican la estructura

- **JOIN-1:** `precio histórico por cultivo` incluido como feature contextual en
  `master_agricola` a pesar de 71% NaN — el Feature Engineering de Fase 3
  imputará por mediana de cultivo y SHAP evaluará su importancia real.
- **JOIN-2:** Subsidios como flag binario `tiene_subsidio_activo` (0/1) para evitar
  ruido estadístico en un lookup de solo 37 filas.
- **JOIN-3 descartado:** Noticias ↔ Precios solo cubría el 22.9% — imputar el 77%
  restante a neutro destruiría la variabilidad del modelo P4.
- **JOIN-5 descartado:** Sin píxeles reales, cruzar metadatos de imágenes con clima
  aporta valor analítico ínfimo para P5.

### Cobertura de los 6 problemas ML del enunciado

| Problema | Dataset principal | Estado |
|---|---|---|
| P1 — Predicción Rendimiento Agrícola | `master_agricola.csv` | Listo |
| P2 — Clasificación Riesgo Plagas | `reportes_plagas_processed.csv` | Listo |
| P3 — Segmentación Países | `master_ganadero.csv` | Listo |
| P4 — Predicción Precios Mercado | `precios_mercado_processed.csv` | Listo |
| P5 — Detección Anomalías Producción | `master_agricola.csv` | Listo |
| P6 — Recomendación de Cultivos | `master_agricola.csv` | Listo |